# Baseline Model — Credit Risk

Train simple models to establish a performance baseline.

In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
import mlflow
import mlflow.sklearn

# Save tracking at project root
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
mlflow.set_tracking_uri(f"sqlite:///{project_root}/mlflow.db")

df = pd.read_csv('../data/processed/credit_risk_clean.csv')
print(f'Shape: {df.shape}')
df.head()

## 1. Split Features and Target

In [2]:
X = df.drop('SeriousDlqin2yrs', axis=1)
y = df['SeriousDlqin2yrs']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {len(X_train)} | Test: {len(X_test)}')
print(f'Default rate train: {y_train.mean():.2%}')
print(f'Default rate test:  {y_test.mean():.2%}')

Train: 119988 | Test: 29998
Default rate train: 6.68%
Default rate test:  6.68%


## 2. Logistic Regression

In [3]:
mlflow.set_experiment('credit-risk')

with mlflow.start_run(run_name='logistic-regression'):
    model_lr = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
    model_lr.fit(X_train, y_train)

    y_pred_lr = model_lr.predict(X_test)
    y_prob_lr = model_lr.predict_proba(X_test)[:, 1]
    auc_lr = roc_auc_score(y_test, y_prob_lr)

    mlflow.log_param('model_type', 'LogisticRegression')
    mlflow.log_param('class_weight', 'balanced')
    mlflow.log_metric('auc', auc_lr)
    mlflow.sklearn.log_model(model_lr, 'model')

    print(f'AUC: {auc_lr:.4f}')
    print()
    print(classification_report(y_test, y_pred_lr, target_names=['Paid', 'Defaulted']))

2026/04/07 11:43:28 INFO mlflow.tracking.fluent: Experiment with name 'credit-risk' does not exist. Creating a new experiment.
/home/adriano/Projetos/ml/ml-finance-agent/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
2026/04/07 11:43:49 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/07 11:43:49 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution becaus

AUC: 0.8619

              precision    recall  f1-score   support

        Paid       0.98      0.80      0.88     27993
   Defaulted       0.21      0.76      0.33      2005

    accuracy                           0.80     29998
   macro avg       0.60      0.78      0.61     29998
weighted avg       0.93      0.80      0.84     29998



## 3. Random Forest

In [4]:
with mlflow.start_run(run_name='random-forest'):
    model_rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
    model_rf.fit(X_train, y_train)

    y_pred_rf = model_rf.predict(X_test)
    y_prob_rf = model_rf.predict_proba(X_test)[:, 1]
    auc_rf = roc_auc_score(y_test, y_prob_rf)

    mlflow.log_param('model_type', 'RandomForestClassifier')
    mlflow.log_param('n_estimators', 100)
    mlflow.log_param('class_weight', 'balanced')
    mlflow.log_metric('auc', auc_rf)
    mlflow.sklearn.log_model(model_rf, 'model')

    print(f'AUC: {auc_rf:.4f}')
    print()
    print(classification_report(y_test, y_pred_rf, target_names=['Paid', 'Defaulted']))

2026/04/07 11:44:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/07 11:44:02 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


AUC: 0.8387

              precision    recall  f1-score   support

        Paid       0.94      0.99      0.97     27993
   Defaulted       0.54      0.16      0.24      2005

    accuracy                           0.93     29998
   macro avg       0.74      0.57      0.60     29998
weighted avg       0.92      0.93      0.92     29998



## 4. Compare Results

In [5]:
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest'],
    'AUC': [auc_lr, auc_rf],
})

print(results.to_string(index=False))
print(f'\nBest model: {results.loc[results["AUC"].idxmax(), "Model"]}')

              Model      AUC
Logistic Regression 0.861944
      Random Forest 0.838731

Best model: Logistic Regression


## 5. Register Best Model

In [6]:
client = mlflow.MlflowClient()
experiment = client.get_experiment_by_name('credit-risk')
runs = client.search_runs(experiment.experiment_id, order_by=['metrics.auc DESC'], max_results=1)
best_run = runs[0]

print(f'Best run: {best_run.info.run_name}')
print(f'AUC: {best_run.data.metrics["auc"]:.4f}')

model_uri = f'runs:/{best_run.info.run_id}/model'
result = mlflow.register_model(model_uri, 'credit-risk-model')
print(f'Registered: {result.name} v{result.version}')

Successfully registered model 'credit-risk-model'.
2026/04/07 11:44:04 WARNING mlflow.tracking._model_registry.fluent: Run with id 8e766045ef914c7daeddf1246dd03737 has no artifacts at artifact path 'model', registering model based on models:/m-5fbd8ffbb0444356a0414cf13d90400b instead


Best run: logistic-regression
AUC: 0.8619
Registered: credit-risk-model v1


Created version '1' of model 'credit-risk-model'.
